# Ventilation EDA Prep

Prepare an hourly ICU ventilation-risk cohort and align the latest vital signs so the dataset is ready for exploratory analysis.

In [1]:
import pandas as pd

## 1) Data source & tables used

**Dataset:** eICU Collaborative Research Database (eICU‑CRD) — demo v2.0.1  
The data is *de-identified* and intended for research/education.

**Tables used in this notebook:**
- `respiratoryCare.csv.gz` → ventilation start/end offsets (`ventstartoffset`, …)
- `patient.csv.gz` → ICU stay metadata (e.g., `unitdischargeoffset`)
- `vitalPeriodic.csv.gz` → time-series vitals with `observationoffset`

**Time unit:** offsets are in **minutes since ICU admission**.


In [2]:


path = "eicu-collaborative-research-database-demo-2.0.1/"

resp_care = pd.read_csv(path + "respiratoryCare.csv.gz")
resp_chart = pd.read_csv(path + "respiratoryCharting.csv.gz")
treat = pd.read_csv(path + "treatment.csv.gz")

resp_care.head()

,respcareid,patientunitstayid,respcarestatusoffset,currenthistoryseqnum,airwaytype,airwaysize,airwayposition,cuffpressure,ventstartoffset,ventendoffset,...,peeplimit,cpaplimit,setapneainterval,setapneatv,setapneaippeephigh,setapnearr,setapneapeakflow,setapneainsptime,setapneaie,setapneafio2
0,564013,147784,1188,2,NaN,NaN,NaN,NaN,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,564012,147784,-61,1,NaN,NaN,NaN,NaN,-361,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,545261,165840,-63,1,NaN,NaN,NaN,NaN,-363,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,545262,165840,73,2,NaN,NaN,NaN,NaN,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,550472,187150,7293,2,NaN,NaN,NaN,NaN,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Inspect respiratoryCare columns

List the available fields to identify the variables needed for ventilation-event extraction.

In [3]:
resp_care.columns

Index(['respcareid', 'patientunitstayid', 'respcarestatusoffset',
       'currenthistoryseqnum', 'airwaytype', 'airwaysize', 'airwayposition',
       'cuffpressure', 'ventstartoffset', 'ventendoffset',
       'priorventstartoffset', 'priorventendoffset', 'apneaparams',
       'lowexhmvlimit', 'hiexhmvlimit', 'lowexhtvlimit', 'hipeakpreslimit',
       'lowpeakpreslimit', 'hirespratelimit', 'lowrespratelimit',
       'sighpreslimit', 'lowironoxlimit', 'highironoxlimit',
       'meanairwaypreslimit', 'peeplimit', 'cpaplimit', 'setapneainterval',
       'setapneatv', 'setapneaippeephigh', 'setapnearr', 'setapneapeakflow',
       'setapneainsptime', 'setapneaie', 'setapneafio2'],
      dtype='str')

### Check respiratoryCare size

Confirm the table dimensions before filtering.

In [4]:
resp_care.shape

(5436, 34)

## 2) Cohort & event definition (label anchor)

We define the deterioration event as:

- **First ventilation start per ICU stay**  
  `vent_start_min = min(ventstartoffset)`

We filter out invalid records:
- remove missing `ventstartoffset`
- remove `ventstartoffset <= 0` (before admission / corrupted)

This gives one event time (or *no event*) for every `patientunitstayid`.


In [5]:


path = "eicu-collaborative-research-database-demo-2.0.1/"

resp = pd.read_csv(path + "respiratoryCare.csv.gz")

# فقط السجلات اللي فيها تهوية
vent = resp[resp["ventstartoffset"].notna()].copy()

# نحذف القيم الغلط (قبل الدخول او صفر)
vent = vent[vent["ventstartoffset"] > 0]

# أول تهوية لكل مريض
vent_start = (
    vent.groupby("patientunitstayid")["ventstartoffset"]
    .min()
    .reset_index()
    .rename(columns={"ventstartoffset": "vent_start_min"})
)

print("عدد المرضى اللي احتاجوا ventilator:", len(vent_start))
vent_start.head()


عدد المرضى اللي احتاجوا ventilator: 69


,patientunitstayid,vent_start_min
0,197619,310
1,202294,433
2,230427,3818
3,325924,1043
4,341782,1075


### Validate event table shape

Quick check of the resulting event-level table.

In [6]:
vent_start.shape

(69, 2)

## 3) Build a patient-hour time grid

To make “early prediction” possible, we convert each ICU stay into **hourly checkpoints**:

For each `patientunitstayid`:
- create `hour = 0..floor(unitdischargeoffset/60)`
- compute `t_min = hour * 60`

So each row represents:  
**“At time t, what is the risk in the next 6–12 hours?”**


In [7]:
patient = pd.read_csv(path + "patient.csv.gz")

[c for c in patient.columns if "offset" in c.lower()]

['hospitaladmitoffset', 'hospitaldischargeoffset', 'unitdischargeoffset']

### Build the patient-hour grid

Use each ICU stay duration to expand one row per patient-hour and create the aligned time variable `t_min`.

In [8]:
stay = patient[["patientunitstayid","unitdischargeoffset"]].copy()

# نحول مدة الإقامة لساعات
def make_hours(max_min):
    max_h = int(max_min // 60)
    return list(range(0, max_h + 1))

hours_df = (
    stay
    .assign(hour=stay["unitdischargeoffset"].apply(make_hours))
    .explode("hour")
    .assign(t_min=lambda d: d["hour"] * 60)
    [["patientunitstayid","hour","t_min"]]
)

hours_df.head(), hours_df.shape

(   patientunitstayid hour t_min
 0             141764    0     0
 0             141764    1    60
 0             141764    2   120
 0             141764    3   180
 0             141764    4   240,
 (147645, 3))

## 4) Label creation (6–12 hours before ventilation)

For each patient-hour at time `t_min`:

- **High risk (y_high=1)** if:
  - ventilation exists **and**
  - `t_min ∈ [vent_start_min - 12h, vent_start_min - 6h]`

### Leakage prevention (important)
To avoid giving the model information too close to (or after) the event, we remove rows:

- **Last 6 hours before event:** `t_min > vent_start_min - 6h`
- **At/after event:** `t_min >= vent_start_min`

This forces the task to be truly “early warning”.


In [9]:
df = hours_df.merge(vent_start, on="patientunitstayid", how="left")

# 1) High risk إذا الوقت داخل [V-12h, V-6h]
df["y_high"] = (
    df["vent_start_min"].notna()
    & (df["t_min"] >= df["vent_start_min"] - 12*60)
    & (df["t_min"] <= df["vent_start_min"] - 6*60)
).astype(int)

# 2) امنع leakage: اشيل أي نافذة قريبة جداً من الحدث (آخر 6 ساعات) وبعد الحدث
df = df[~(df["vent_start_min"].notna() & (df["t_min"] > df["vent_start_min"] - 6*60))]
df = df[~(df["vent_start_min"].notna() & (df["t_min"] >= df["vent_start_min"]))]

print(df["y_high"].value_counts())

y_high
0    136581
1       298
Name: count, dtype: int64


### Preview the cohort table

Display the labeled patient-hour cohort after merging and filtering.

In [10]:
df

,patientunitstayid,hour,t_min,vent_start_min,y_high
0,141764,0,0,NaN,0
1,141764,1,60,NaN,0
2,141764,2,120,NaN,0
3,141764,3,180,NaN,0
4,141764,4,240,NaN,0
...,...,...,...,...,...
147640,3353113,40,2400,NaN,0
147641,3353113,41,2460,NaN,0
147642,3353113,42,2520,NaN,0
147643,3353113,43,2580,NaN,0


### Run cohort quality checks

Check shape, uniqueness, missingness, and label validity for the cohort table.

In [11]:
# basic checks
print("shape:", df.shape)
print("duplicate patient-hour rows:", df.duplicated(subset=["patientunitstayid", "hour"]).sum())
print("negative t_min rows:", (df["t_min"] < 0).sum())
print("missing patient IDs:", df["patientunitstayid"].isna().sum())
print("missing y_high:", df["y_high"].isna().sum())
print("invalid y_high values:", (~df["y_high"].isin([0, 1])).sum())

shape: (136879, 5)
duplicate patient-hour rows: 0
negative t_min rows: 0
missing patient IDs: 0
missing y_high: 0
invalid y_high values: 0


### Validate event-label consistency

Confirm that event times are valid and that no positive labels exist without an associated ventilation event.

In [12]:
print(
    "invalid vent_start_min <= 0:",
    ((df["vent_start_min"].notna()) & (df["vent_start_min"] <= 0)).sum()
)

print(
    "positive rows without event:",
    ((df["y_high"] == 1) & (df["vent_start_min"].isna())).sum()
)

invalid vent_start_min <= 0: 0
positive rows without event: 0


### Section break

This cell is currently empty and can be used later for notes or additional checks.

## 5) Vitals (initial features) — plan

We start by loading vitals from `vitalPeriodic`:
- `heartrate`, `respiration`, `sao2` (SpO₂)

**Next step:** aggregate vitals into features aligned with the hourly grid, for example:
- last observed value within the last hour
- mean/min/max over the last **6 hours**
- trend (slope) over the last **6 hours**
- variability (std) over the last **6 hours**

> Trend features often outperform raw values in deterioration prediction.


In [13]:
vital = pd.read_csv(path + "vitalPeriodic.csv.gz", nrows=5)
print(vital.columns)

Index(['vitalperiodicid', 'patientunitstayid', 'observationoffset',
       'temperature', 'sao2', 'heartrate', 'respiration', 'cvp', 'etco2',
       'systemicsystolic', 'systemicdiastolic', 'systemicmean', 'pasystolic',
       'padiastolic', 'pamean', 'st1', 'st2', 'st3', 'icp'],
      dtype='str')


### Load vitals and inspect candidate fields

Read the full vital table and list the relevant variables for heart rate, respiration, oxygen saturation, and timing.

In [14]:
vital = pd.read_csv(path + "vitalPeriodic.csv.gz")
[v for v in vital.columns if "heart" in v.lower() or "spo2" in v.lower() or "resp" in v.lower() or "map" in v.lower() or "offset" in v.lower()]

['observationoffset', 'heartrate', 'respiration']

### Keep required vital columns

Load only the columns needed for alignment and drop rows missing the observation time.

In [15]:
vital = pd.read_csv(path + "vitalPeriodic.csv.gz",
                    usecols=["patientunitstayid","observationoffset","heartrate","respiration","sao2"])

vital = vital.dropna(subset=["observationoffset"])
vital.head()

,patientunitstayid,observationoffset,sao2,heartrate,respiration
0,141765,1179,NaN,82.0,NaN
1,141765,189,97.0,76.0,30.0
2,141765,1169,NaN,84.0,NaN
3,141765,1534,NaN,92.0,NaN
4,141765,1164,NaN,86.0,NaN


### Check vital table structure

Confirm the shape and columns of the reduced vital table.

In [16]:
print(vital.shape)
print(vital.columns.tolist())
vital.head()

(1634960, 5)
['patientunitstayid', 'observationoffset', 'sao2', 'heartrate', 'respiration']


,patientunitstayid,observationoffset,sao2,heartrate,respiration
0,141765,1179,NaN,82.0,NaN
1,141765,189,97.0,76.0,30.0
2,141765,1169,NaN,84.0,NaN
3,141765,1534,NaN,92.0,NaN
4,141765,1164,NaN,86.0,NaN


### Clean the vital table

Remove duplicates, enforce valid observation offsets, and replace physiologically implausible values with missing values.

In [17]:
vital = vital.copy()

# remove duplicates
vital = vital.drop_duplicates()

# remove missing key fields
vital = vital.dropna(subset=["patientunitstayid", "observationoffset"])

# keep only valid time offsets
vital = vital[vital["observationoffset"] >= 0]

# clean impossible values
vital.loc[(vital["heartrate"] < 20) | (vital["heartrate"] > 250), "heartrate"] = pd.NA
vital.loc[(vital["respiration"] < 4) | (vital["respiration"] > 80), "respiration"] = pd.NA
vital.loc[(vital["sao2"] < 40) | (vital["sao2"] > 100), "sao2"] = pd.NA

# type fixes
vital["patientunitstayid"] = vital["patientunitstayid"].astype(int)
vital["observationoffset"] = vital["observationoffset"].astype(int)

print("shape:", vital.shape)
print("duplicate rows:", vital.duplicated().sum())
print("negative offsets:", (vital["observationoffset"] < 0).sum())
print(vital.isna().mean())

vital.head()

shape: (1633310, 5)
duplicate rows: 0
negative offsets: 0
patientunitstayid    0.000000
observationoffset    0.000000
sao2                 0.119326
heartrate            0.004535
respiration          0.163167
dtype: float64


,patientunitstayid,observationoffset,sao2,heartrate,respiration
0,141765,1179,NaN,82.0,NaN
1,141765,189,97.0,76.0,30.0
2,141765,1169,NaN,84.0,NaN
3,141765,1534,NaN,92.0,NaN
4,141765,1164,NaN,86.0,NaN


### Prepare feature tables for alignment

Create the base cohort table and separate heart-rate, respiration, and oxygen-saturation tables that will later be aligned to each patient-hour row.

In [18]:
df_base = df.sort_values(["patientunitstayid", "t_min"]).reset_index(drop=True).copy()

v_hr = (
    vital[["patientunitstayid", "observationoffset", "heartrate"]]
    .dropna(subset=["heartrate"])
    .sort_values(["patientunitstayid", "observationoffset"])
    .reset_index(drop=True)
)

v_rr = (
    vital[["patientunitstayid", "observationoffset", "respiration"]]
    .dropna(subset=["respiration"])
    .sort_values(["patientunitstayid", "observationoffset"])
    .reset_index(drop=True)
)

v_sao2 = (
    vital[["patientunitstayid", "observationoffset", "sao2"]]
    .dropna(subset=["sao2"])
    .sort_values(["patientunitstayid", "observationoffset"])
    .reset_index(drop=True)
)

### Fix merge key data types

Convert merge keys to numeric types, align the `patientunitstayid` dtype on both sides, and drop rows that became invalid during conversion.

In [20]:
df_base = df_base.copy()
v_hr = v_hr.copy()

# make merge keys numeric
df_base["t_min"] = pd.to_numeric(df_base["t_min"], errors="coerce")
v_hr["observationoffset"] = pd.to_numeric(v_hr["observationoffset"], errors="coerce")

# make by-key same type
df_base["patientunitstayid"] = pd.to_numeric(df_base["patientunitstayid"], errors="coerce").astype("Int64")
v_hr["patientunitstayid"] = pd.to_numeric(v_hr["patientunitstayid"], errors="coerce").astype("Int64")

# drop rows that became missing after conversion
df_base = df_base.dropna(subset=["patientunitstayid", "t_min"])
v_hr = v_hr.dropna(subset=["patientunitstayid", "observationoffset"])

# convert to regular int
df_base["patientunitstayid"] = df_base["patientunitstayid"].astype(int)
v_hr["patientunitstayid"] = v_hr["patientunitstayid"].astype(int)

df_base["t_min"] = df_base["t_min"].astype(int)
v_hr["observationoffset"] = v_hr["observationoffset"].astype(int)

# sort both sides
df_base = df_base.sort_values(["patientunitstayid", "t_min"]).reset_index(drop=True)
v_hr = v_hr.sort_values(["patientunitstayid", "observationoffset"]).reset_index(drop=True)

print(df_base.dtypes)
print(v_hr.dtypes)

patientunitstayid      int64
hour                  object
t_min                  int64
vent_start_min       float64
y_high                 int64
dtype: object
patientunitstayid      int64
observationoffset      int64
heartrate            float64
dtype: object


### Sort keys for as-of merge

Sort both tables so `merge_asof` can align each patient-hour with the latest prior observation.

In [22]:
df_base = df_base.sort_values(["t_min", "patientunitstayid"]).reset_index(drop=True)
v_hr = v_hr.sort_values(["observationoffset", "patientunitstayid"]).reset_index(drop=True)

### Verify sort order

Inspect the first rows and confirm that the merge keys are monotonically increasing.

In [23]:
print(df_base[["patientunitstayid", "t_min"]].head(10))
print(v_hr[["patientunitstayid", "observationoffset"]].head(10))

print(df_base["t_min"].is_monotonic_increasing)
print(v_hr["observationoffset"].is_monotonic_increasing)

   patientunitstayid  t_min
0             141764      0
1             141765      0
2             143870      0
3             144815      0
4             145427      0
5             147306      0
6             147307      0
7             147784      0
8             148611      0
9             149432      0
   patientunitstayid  observationoffset
0             187150                  0
1             263285                  0
2             496831                  0
3            1008970                  0
4            1015158                  0
5            1036759                  0
6            1054428                  0
7            1083722                  0
8            1096127                  0
9            1103224                  0
True
True


### Attach latest heart rate

Use an as-of merge to map the most recent prior heart-rate value to each patient-hour row.

In [24]:
eda_ready_df = pd.merge_asof(
    df_base,
    v_hr,
    left_on="t_min",
    right_on="observationoffset",
    by="patientunitstayid",
    direction="backward"
)

eda_ready_df = eda_ready_df.rename(columns={"heartrate": "hr_last"})
eda_ready_df = eda_ready_df.drop(columns=["observationoffset"])

### Prepare respiration and SpO2 tables

Create and type-clean the respiration and oxygen-saturation tables before the remaining as-of merges.

In [26]:
v_rr = vital[["patientunitstayid", "observationoffset", "respiration"]].dropna(subset=["respiration"]).copy()
v_rr["patientunitstayid"] = pd.to_numeric(v_rr["patientunitstayid"], errors="coerce")
v_rr["observationoffset"] = pd.to_numeric(v_rr["observationoffset"], errors="coerce")
v_rr = v_rr.dropna(subset=["patientunitstayid", "observationoffset"])
v_rr["patientunitstayid"] = v_rr["patientunitstayid"].astype(int)
v_rr["observationoffset"] = v_rr["observationoffset"].astype(int)
v_rr = v_rr.sort_values(["observationoffset", "patientunitstayid"]).reset_index(drop=True)

v_sao2 = vital[["patientunitstayid", "observationoffset", "sao2"]].dropna(subset=["sao2"]).copy()
v_sao2["patientunitstayid"] = pd.to_numeric(v_sao2["patientunitstayid"], errors="coerce")
v_sao2["observationoffset"] = pd.to_numeric(v_sao2["observationoffset"], errors="coerce")
v_sao2 = v_sao2.dropna(subset=["patientunitstayid", "observationoffset"])
v_sao2["patientunitstayid"] = v_sao2["patientunitstayid"].astype(int)
v_sao2["observationoffset"] = v_sao2["observationoffset"].astype(int)
v_sao2 = v_sao2.sort_values(["observationoffset", "patientunitstayid"]).reset_index(drop=True)

### Attach latest respiration

Map the latest prior respiration value to each patient-hour row.

In [27]:
eda_ready_df = eda_ready_df.sort_values(["t_min", "patientunitstayid"]).reset_index(drop=True)

eda_ready_df = pd.merge_asof(
    eda_ready_df,
    v_rr,
    left_on="t_min",
    right_on="observationoffset",
    by="patientunitstayid",
    direction="backward"
)

eda_ready_df = eda_ready_df.rename(columns={"respiration": "rr_last"})
eda_ready_df = eda_ready_df.drop(columns=["observationoffset"])

### Attach latest SpO2

Map the latest prior oxygen-saturation value to each patient-hour row.

In [28]:
eda_ready_df = eda_ready_df.sort_values(["t_min", "patientunitstayid"]).reset_index(drop=True)

eda_ready_df = pd.merge_asof(
    eda_ready_df,
    v_sao2,
    left_on="t_min",
    right_on="observationoffset",
    by="patientunitstayid",
    direction="backward"
)

eda_ready_df = eda_ready_df.rename(columns={"sao2": "sao2_last"})
eda_ready_df = eda_ready_df.drop(columns=["observationoffset"])

### Add missingness indicators

Create binary flags that show whether each latest vital sign is missing at a given patient-hour.

In [29]:
eda_ready_df["hr_missing"] = eda_ready_df["hr_last"].isna().astype(int)
eda_ready_df["rr_missing"] = eda_ready_df["rr_last"].isna().astype(int)
eda_ready_df["sao2_missing"] = eda_ready_df["sao2_last"].isna().astype(int)

### Final cleanup

Normalize `hour`, remove invalid rows if any were introduced, and enforce one row per patient-hour.

In [30]:
eda_ready_df["hour"] = pd.to_numeric(eda_ready_df["hour"], errors="coerce")
eda_ready_df = eda_ready_df.dropna(subset=["hour"])
eda_ready_df["hour"] = eda_ready_df["hour"].astype(int)

eda_ready_df = eda_ready_df.drop_duplicates(subset=["patientunitstayid", "hour"]).reset_index(drop=True)

### Preview the final EDA-ready table

Inspect the final shape, columns, and first rows of the prepared dataset.

In [31]:
print(eda_ready_df.shape)
print(eda_ready_df.columns.tolist())
eda_ready_df.head()

(136879, 11)
['patientunitstayid', 'hour', 't_min', 'vent_start_min', 'y_high', 'hr_last', 'rr_last', 'sao2_last', 'hr_missing', 'rr_missing', 'sao2_missing']


,patientunitstayid,hour,t_min,vent_start_min,y_high,hr_last,rr_last,sao2_last,hr_missing,rr_missing,sao2_missing
0,141764,0,0,NaN,0,NaN,NaN,NaN,1,1,1
1,141765,0,0,NaN,0,NaN,NaN,NaN,1,1,1
2,143870,0,0,NaN,0,NaN,NaN,NaN,1,1,1
3,144815,0,0,NaN,0,NaN,NaN,NaN,1,1,1
4,145427,0,0,NaN,0,NaN,NaN,NaN,1,1,1


### Final data-quality summary

Verify uniqueness and target completeness, then summarize missingness rates for the aligned vital features.

In [32]:
print("duplicate patient-hour rows:", eda_ready_df.duplicated(subset=["patientunitstayid", "hour"]).sum())
print("missing target:", eda_ready_df["y_high"].isna().sum())
print(eda_ready_df[["hr_missing", "rr_missing", "sao2_missing"]].mean())

duplicate patient-hour rows: 0
missing target: 0
hr_missing      0.046172
rr_missing      0.152405
sao2_missing    0.058088
dtype: float64


### Save the prepared dataset

Export the final EDA-ready table to CSV for later analysis.

In [33]:
eda_ready_df.to_csv("ventilation_eda_ready.csv", index=False)

## 6) Evaluation plan (when modeling starts)

Because deterioration events are *rare*, use:
- **AUPRC** (most important here)
- AUROC
- Sensitivity at fixed specificity (clinical-friendly)

**Data split must be patient-level** (no leakage across time windows of the same patient):
- train/val/test split by `patientunitstayid`

Also report **lead time**:
- how many hours before the event the model starts triggering consistently.


## 7) Portfolio deliverables

- ✅ Notebook (this): cohort + label creation + leakage-safe framing
- ⏭️ Next: feature engineering + baseline models (LogReg/XGBoost)
- ⏭️ Advanced: sequence models (LSTM/GRU / TCN / Transformer)
- ⏭️ Interpretability: **SHAP** for top risk drivers
- ⏭️ Simple Streamlit demo:
  - choose patient → show current risk + top factors + trend chart
